# Notebook 04 — MixUp Augmentation

**Session:** Image Augmentation  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement **MixUp** (Zhang et al., 2018): construct convex linear combinations of paired training images $\tilde{x} = \lambda x_i + (1 - \lambda) x_j$ with $\lambda \sim \text{Beta}(\alpha, \alpha)$ and optimize the interpolated cross-entropy objective.

**Learning objectives:**
- Implement `mixup_data` and `mixup_criterion` cleanly for any device.
- Pass the mixed tensor `mixed_inputs` through the network during training.
- Observe regularized training vs. clean validation accuracy.

In [ ]:
from pathlib import Path

import albumentations
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18

## 1  MixUp Data Blending & Loss Criterion

In [ ]:
def get_accuracy(model: nn.Module, data_loader: DataLoader, device: torch.device) -> float:
    """Compute classification accuracy (%) over an entire DataLoader."""
    correct = 0
    total = 0

    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100.0 * (correct / total) if total > 0 else 0.0


def plot_losses(
    train_acc: list[float],
    valid_acc: list[float],
    train_loss: list[float],
    valid_loss: list[float],
) -> None:
    """Plot training and validation accuracy and loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(train_acc, color="tab:blue", label="Train Accuracy")
    ax1.plot(valid_acc, color="tab:red", label="Validation Accuracy")
    ax1.set(title="Accuracy over Epochs", xlabel="Epoch", ylabel="Accuracy (%)")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.plot(train_loss, color="tab:blue", label="Train Loss")
    ax2.plot(valid_loss, color="tab:red", label="Validation Loss")
    ax2.set(title="Loss over Epochs", xlabel="Epoch", ylabel="Loss")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    fig.tight_layout()
    plt.show()


def validate(
    valid_loader: DataLoader,
    model: nn.Module,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[nn.Module, float]:
    """Run one validation epoch without gradient tracking."""
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

    epoch_loss = running_loss / max(len(valid_loader), 1)
    return model, epoch_loss


def mixup_data(
    x: torch.Tensor,
    y: torch.Tensor,
    alpha: float = 1.0,
    device: torch.device | str = "cpu",
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """Return convexly mixed inputs, paired labels, and mixing coefficient lambda."""
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=device)

    mixed_x = lam * x + (1.0 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(
    criterion: nn.Module,
    pred: torch.Tensor,
    y_a: torch.Tensor,
    y_b: torch.Tensor,
    lam: float,
) -> torch.Tensor:
    """Convex combination of cross-entropy losses for mixed targets."""
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)


def train_mixup(
    train_loader: DataLoader,
    model: nn.Module,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    alpha: float = 1.0,
) -> tuple[nn.Module, torch.optim.Optimizer, float]:
    """Run one training epoch using MixUp blended inputs."""
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        mixed_inputs, targets_a, targets_b, lam = mixup_data(images, labels, alpha, device)

        outputs = model(mixed_inputs)
        loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        running_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model, optimizer, running_loss / max(len(train_loader), 1)


def training_loop(
    model: nn.Module,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    epochs: int,
    device: torch.device,
) -> tuple[nn.Module, torch.optim.Optimizer, tuple[list[float], list[float]]]:
    """Train with MixUp and visualize training/validation curves."""
    train_losses: list[float] = []
    valid_losses: list[float] = []
    list_train_acc: list[float] = []
    list_val_acc: list[float] = []

    for epoch in range(epochs):
        model, optimizer, train_loss = train_mixup(
            train_loader, model, criterion, optimizer, device
        )
        model, valid_loss = validate(valid_loader, model, criterion, device)
        train_acc = get_accuracy(model, train_loader, device=device)
        valid_acc = get_accuracy(model, valid_loader, device=device)

        print(
            f"Epoch {epoch:02d} | Train loss: {train_loss:.4f} | Valid loss: {valid_loss:.4f} "
            f"| Train acc: {train_acc:.2f}% | Valid acc: {valid_acc:.2f}%"
        )
        list_train_acc.append(train_acc)
        list_val_acc.append(valid_acc)
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

    plot_losses(list_train_acc, list_val_acc, train_losses, valid_losses)
    return model, optimizer, (train_losses, valid_losses)

## 2  Dataset, Model, and Execution

In [ ]:
class DogCatDataset(Dataset):
    """Portable image classification dataset reading train/test class subfolders."""

    def __init__(self, root_dir: str | Path, transform=None) -> None:
        self.root_dir = Path(root_dir)
        self.list_images_path: list[Path] = []
        self.list_labels: list[str] = []
        self.class_to_idx = {"dogs": 0, "cats": 1}
        self.transform = transform

        for sub_dir in sorted(self.root_dir.iterdir()):
            if not sub_dir.is_dir() or sub_dir.name not in self.class_to_idx:
                continue
            for image_path in sorted(sub_dir.iterdir()):
                if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.list_images_path.append(image_path)
                    self.list_labels.append(sub_dir.name)

    def __len__(self) -> int:
        return len(self.list_images_path)

    def __getitem__(self, idx: int) -> tuple[np.ndarray, np.ndarray]:
        image = cv2.imread(str(self.list_images_path[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))
        label = np.array(self.class_to_idx[self.list_labels[idx]], dtype=np.int64)

        if self.transform is not None:
            res = self.transform(image=image)
            image = res["image"].astype(np.float32)
        else:
            image = image.astype(np.float32) / 255.0

        image = image.transpose(2, 0, 1)
        return image, label

In [ ]:
class Net(nn.Module):
    """Lightweight ResNet-18 classifier with a 3x3 input stem for 2-class tasks."""

    def __init__(self, num_classes: int = 2) -> None:
        super().__init__()
        self.model = resnet18(weights=None, num_classes=num_classes)
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


DATA_DIR = Path("../data/sample_dataset")
norm_transform = albumentations.Compose([albumentations.Normalize()])

train_data = DogCatDataset(DATA_DIR / "train", transform=norm_transform)
test_data = DogCatDataset(DATA_DIR / "test", transform=norm_transform)

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = DataLoader(test_data, batch_size=8, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

model, optimizer, _ = training_loop(
    model, criterion, optimizer, train_loader, test_loader, epochs=5, device=device
)